In [4]:
# ============================================================
# Near-Duplicate Detection – Perceptual Hashing
# ============================================================
# pip install imagehash pillow pandas tqdm

import os
import imagehash
import pandas as pd
from PIL import Image
from tqdm import tqdm

IMAGE_FOLDER = r'C:\Users\Lukas\masterthesis\thesis_2026\data\manual_sample_images'
OUTPUT_CSV   = r'C:\Users\Lukas\masterthesis\thesis_2026\data\duplicates.csv'

SUPPORTED_FORMATS = ('.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG')
THRESHOLD = 10  # ← je kleiner, desto strenger (0 = identisch, 10 = sehr ähnlich)

image_paths = [
    os.path.join(IMAGE_FOLDER, f)
    for f in os.listdir(IMAGE_FOLDER)
    if f.endswith(SUPPORTED_FORMATS)
]

print(f'Bilder gefunden: {len(image_paths)}')

# Hashes berechnen
hashes = {}
errors = []

for img_path in tqdm(image_paths, desc='Hashing'):
    try:
        img  = Image.open(img_path).convert('RGB')
        h    = imagehash.phash(img)
        hashes[img_path] = h
    except Exception as e:
        errors.append({'filepath': img_path, 'error': str(e)})

print(f'✅ {len(hashes)} Hashes berechnet')
print(f'⚠️  {len(errors)} Fehler')

# Duplikate finden
paths  = list(hashes.keys())
keep   = set(paths)  # alle behalten zunächst
pairs  = []

for i in range(len(paths)):
    for j in range(i + 1, len(paths)):
        dist = hashes[paths[i]] - hashes[paths[j]]
        if dist <= THRESHOLD:
            pairs.append({
                'image_1': os.path.basename(paths[i]),
                'image_2': os.path.basename(paths[j]),
                'distance': dist
            })
            # Das zweite Bild als Duplikat markieren
            keep.discard(paths[j])

print(f'\n✅ Duplikat-Paare gefunden: {len(pairs)}')
print(f'✅ Bilder nach Filterung:   {len(keep)}')
print(f'✅ Entfernt:                {len(paths) - len(keep)}')

# Ergebnisse speichern
df_pairs = pd.DataFrame(pairs)
df_pairs.to_csv(OUTPUT_CSV, index=False)
print(f'✅ CSV gespeichert: {OUTPUT_CSV}')

# Gefilterte Liste speichern
df_keep = pd.DataFrame({
    'filename': [os.path.basename(p) for p in keep],
    'filepath': list(keep)
})
df_keep.to_csv(OUTPUT_CSV.replace('.csv', '_clean.csv'), index=False)
print(f'✅ Gefilterte Liste: {OUTPUT_CSV.replace(".csv", "_clean.csv")}')

Bilder gefunden: 300


Hashing: 100%|███████████████████████████████████████████████████████████████████████| 300/300 [00:15<00:00, 19.27it/s]


✅ 300 Hashes berechnet
⚠️  0 Fehler

✅ Duplikat-Paare gefunden: 3
✅ Bilder nach Filterung:   297
✅ Entfernt:                3
✅ CSV gespeichert: C:\Users\Lukas\masterthesis\thesis_2026\data\duplicates.csv
✅ Gefilterte Liste: C:\Users\Lukas\masterthesis\thesis_2026\data\duplicates_clean.csv
